# Apache Spark - DataFrames

**Materia:** Manejo Masivo de Datos (Big Data)

**Profesor:** Javier Moya

**Alumno:**  Asael Rodriguez Leon

## Objetivo
Comprender y aplicar las operaciones de DataFrames
en Apache Spark mediante ejercicios prácticos y
un proyecto de análisis de ventas.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Practica-DataFrames") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Version de Spark:", spark.version)
print("Entorno iniciado correctamente")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 00:03:56 WARN Utils: Your hostname, asael-rodriguez-RS, resolves to a loopback address: 127.0.1.1; using 192.168.0.109 instead (on interface wlp4s0)
26/11/25 00:03:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 00:03:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version de Spark: 4.3.0
Entorno iniciado correctamente


## Ejercicio 1: Creación y exploración

Se crea un DataFrame con cinco empleados y cuatro
columnas: nombre, edad, departamento y salario.
Se consulta su esquema, las primeras tres filas
y el total de empleados registrados.

In [2]:
datos = [
    ("Ana", 28, "Ventas", 35000),
    ("Juan", 35, "IT", 45000),
    ("María", 42, "IT", 55000),
    ("Pedro", 31, "Ventas", 38000),
    ("Laura", 29, "Marketing", 40000)
]

columnas = ["nombre", "edad", "departamento", "salario"]

df = spark.createDataFrame(datos, columnas)

print("ESQUEMA DEL DATAFRAME")
df.printSchema()

print("PRIMERAS 3 FILAS")
df.show(3)

print("Total de empleados:", df.count())

ESQUEMA DEL DATAFRAME
root
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)

PRIMERAS 3 FILAS
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+
only showing top 3 rows
Total de empleados: 5


## Ejercicio 2: Filtrado

Se aplican filtros al DataFrame de empleados
utilizando operadores lógicos para seleccionar
registros según su departamento, salario y edad.

In [3]:
# Empleados de IT con salario mayor a 40000
print("1. Empleados de IT con salario > 40000")

df.filter(
    (F.col("departamento") == "IT") &
    (F.col("salario") > 40000)
).show()

# Empleados de Ventas o Marketing
print("2. Empleados de Ventas o Marketing")

df.filter(
    (F.col("departamento") == "Ventas") |
    (F.col("departamento") == "Marketing")
).show()

# Empleados que no pertenecen a IT
print("3. Empleados que NO son de IT")

df.filter(
    ~(F.col("departamento") == "IT")
).show()

# Empleados entre 30 y 40 años
print("4. Empleados entre 30 y 40 años")

df.filter(
    (F.col("edad") >= 30) &
    (F.col("edad") <= 40)
).show()

1. Empleados de IT con salario > 40000
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+

2. Empleados de Ventas o Marketing
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

3. Empleados que NO son de IT
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

4. Empleados entre 30 y 40 años
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| Pedro|  31|      Ve

## Ejercicio 3: Manipulación de columnas

Se agrega una columna llamada salario_en_5_años
considerando un incremento anual simple del 5%
durante cinco años. Posteriormente, se renombra
la columna nombre por empleado.

In [4]:
df_modificado = df.withColumn(
    "salario_en_5_años",
    F.col("salario") + (F.col("salario") * 0.05 * 5)
)

df_modificado = df_modificado.withColumnRenamed(
    "nombre", "empleado"
)

print("DATAFRAME MODIFICADO")
df_modificado.show()

print("NUEVO ESQUEMA")
df_modificado.printSchema()

DATAFRAME MODIFICADO
+--------+----+------------+-------+-----------------+
|empleado|edad|departamento|salario|salario_en_5_años|
+--------+----+------------+-------+-----------------+
|     Ana|  28|      Ventas|  35000|          43750.0|
|    Juan|  35|          IT|  45000|          56250.0|
|   María|  42|          IT|  55000|          68750.0|
|   Pedro|  31|      Ventas|  38000|          47500.0|
|   Laura|  29|   Marketing|  40000|          50000.0|
+--------+----+------------+-------+-----------------+

NUEVO ESQUEMA
root
 |-- empleado: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)
 |-- salario_en_5_años: double (nullable = true)



## Ejercicio 4: Agregaciones

Se agrupan los empleados por departamento para
calcular el salario promedio, máximo y mínimo,
la cantidad de empleados y el gasto total en
salarios, ordenado de mayor a menor.

In [5]:
# 1. Salario promedio por departamento
print("SALARIO PROMEDIO POR DEPARTAMENTO")

df.groupBy("departamento").agg(
    F.avg("salario").alias("salario_promedio")
).show()

# 2. Salario máximo y mínimo
print("SALARIO MAXIMO Y MINIMO")

df.groupBy("departamento").agg(
    F.max("salario").alias("salario_maximo"),
    F.min("salario").alias("salario_minimo")
).show()

# 3. Cantidad de empleados por departamento
print("EMPLEADOS POR DEPARTAMENTO")

df.groupBy("departamento").count().show()

# 4. Gasto total en salarios
print("GASTO TOTAL EN SALARIOS")

df.groupBy("departamento").agg(
    F.sum("salario").alias("gasto_total")
).orderBy(F.desc("gasto_total")).show()

SALARIO PROMEDIO POR DEPARTAMENTO
+------------+----------------+
|departamento|salario_promedio|
+------------+----------------+
|      Ventas|         36500.0|
|          IT|         50000.0|
|   Marketing|         40000.0|
+------------+----------------+

SALARIO MAXIMO Y MINIMO
+------------+--------------+--------------+
|departamento|salario_maximo|salario_minimo|
+------------+--------------+--------------+
|      Ventas|         38000|         35000|
|          IT|         55000|         45000|
|   Marketing|         40000|         40000|
+------------+--------------+--------------+

EMPLEADOS POR DEPARTAMENTO
+------------+-----+
|departamento|count|
+------------+-----+
|      Ventas|    2|
|          IT|    2|
|   Marketing|    1|
+------------+-----+

GASTO TOTAL EN SALARIOS
+------------+-----------+
|departamento|gasto_total|
+------------+-----------+
|          IT|     100000|
|      Ventas|      73000|
|   Marketing|      40000|
+------------+-----------+



## Ejercicio 5: Window Functions

Se crea un DataFrame de ventas mensuales y se
utiliza una función de ventana para generar un
ranking de productos por mes, ordenando las
ventas de mayor a menor.

In [6]:
# Datos de ventas mensuales
ventas_mensuales = [
    ("Enero", "ProdA", 100),
    ("Enero", "ProdB", 150),
    ("Febrero", "ProdA", 120)
]

df_ventas = spark.createDataFrame(
    ventas_mensuales,
    ["mes", "producto", "ventas"]
)

# Definir la ventana por mes
ventana = Window.partitionBy("mes").orderBy(
    F.col("ventas").desc()
)

# Generar ranking de productos
df_ranking = df_ventas.withColumn(
    "ranking",
    F.rank().over(ventana)
)

print("RANKING DE PRODUCTOS POR MES")
df_ranking.show()

RANKING DE PRODUCTOS POR MES
+-------+--------+------+-------+
|    mes|producto|ventas|ranking|
+-------+--------+------+-------+
|  Enero|   ProdB|   150|      1|
|  Enero|   ProdA|   100|      2|
|Febrero|   ProdA|   120|      1|
+-------+--------+------+-------+



## Ejercicio 6: Joins

Se crean dos DataFrames, uno de personas y otro
de departamentos. Se utiliza un Left Join para
relacionar ambos mediante la columna DepID,
conservando todas las personas registradas.

In [7]:
# DataFrame de personas
personas = spark.createDataFrame([
    ("Ana", 1),
    ("Luis", 2),
    ("Marta", 3)
], ["Nombre", "DepID"])

# DataFrame de departamentos
departamentos = spark.createDataFrame([
    (1, "IT"),
    (2, "HR")
], ["DepID", "NombreDep"])

# Mostrar DataFrames originales
print("PERSONAS")
personas.show()

print("DEPARTAMENTOS")
departamentos.show()

# Realizar Left Join
df_join = personas.join(
    departamentos,
    "DepID",
    "left"
)

print("RESULTADO DEL LEFT JOIN")
df_join.select("Nombre", "DepID", "NombreDep").show()

PERSONAS
+------+-----+
|Nombre|DepID|
+------+-----+
|   Ana|    1|
|  Luis|    2|
| Marta|    3|
+------+-----+

DEPARTAMENTOS
+-----+---------+
|DepID|NombreDep|
+-----+---------+
|    1|       IT|
|    2|       HR|
+-----+---------+

RESULTADO DEL LEFT JOIN
+------+-----+---------+
|Nombre|DepID|NombreDep|
+------+-----+---------+
|   Ana|    1|       IT|
|  Luis|    2|       HR|
| Marta|    3|     NULL|
+------+-----+---------+



### Interpretación del resultado

Marta permanece en el DataFrame resultante porque
el Left Join conserva todos los registros de la
tabla izquierda. Sin embargo, su departamento
aparece como NULL debido a que el DepID 3 no
existe en la tabla de departamentos.

## Ejercicio 7: SQL Puro

Se registra el DataFrame de empleados como una
vista temporal para realizar una consulta SQL
que seleccione a los empleados del departamento
de IT con un salario superior a 40000.

In [8]:
# Registrar el DataFrame como vista temporal
df.createOrReplaceTempView("empleados")

# Ejecutar consulta SQL
resultado_sql = spark.sql("""
    SELECT nombre, edad, departamento, salario
    FROM empleados
    WHERE departamento = 'IT'
    AND salario > 40000
""")

print("RESULTADO DE LA CONSULTA SQL")
resultado_sql.show()

RESULTADO DE LA CONSULTA SQL
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+



## Ejercicio 8: Funciones de String

Se eliminan los espacios en blanco al inicio y
al final de los nombres y apellidos mediante
la función trim. Posteriormente, se concatenan
ambos campos para generar el nombre completo.

In [9]:
# Crear DataFrame con espacios innecesarios
datos_nombres = [
    (" Juan ", "García"),
    ("Ana", " Pérez ")
]

df_nombres = spark.createDataFrame(
    datos_nombres,
    ["Nombre", "Apellido"]
)

print("DATOS ORIGINALES")
df_nombres.show()

# Eliminar espacios en blanco
df_limpio = df_nombres.withColumn(
    "Nombre", F.trim(F.col("Nombre"))
).withColumn(
    "Apellido", F.trim(F.col("Apellido"))
)

# Concatenar nombre y apellido
df_completo = df_limpio.withColumn(
    "Nombre_Completo",
    F.concat_ws(" ", "Nombre", "Apellido")
)

print("DATOS LIMPIOS Y CONCATENADOS")
df_completo.show(truncate=False)

DATOS ORIGINALES
+------+--------+
|Nombre|Apellido|
+------+--------+
| Juan |  García|
|   Ana|  Pérez |
+------+--------+

DATOS LIMPIOS Y CONCATENADOS
+------+--------+---------------+
|Nombre|Apellido|Nombre_Completo|
+------+--------+---------------+
|Juan  |García  |Juan García    |
|Ana   |Pérez   |Ana Pérez      |
+------+--------+---------------+



# Proyecto: Análisis de ventas

## Objetivo

Analizar el rendimiento de las ventas por país
mediante DataFrames de Apache Spark.

Se utiliza el conjunto de datos pdi_sales.csv,
que contiene información de productos, fechas,
códigos postales, unidades vendidas, ingresos
y países.

## 1. Carga y exploración del dataset

Se carga el archivo CSV utilizando punto y coma
como separador, reconociendo las cabeceras e
infiriendo automáticamente los tipos de datos.
Se muestra el esquema y el total de registros.

In [11]:
# Cargar dataset de ventas
df_sales = spark.read.csv(
    "pdi_sales.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

print("ESQUEMA DEL DATASET")
df_sales.printSchema()

print("PRIMEROS 5 REGISTROS")
df_sales.show(5, truncate=False)

print("TOTAL DE REGISTROS:", df_sales.count())
print("TOTAL DE COLUMNAS:", len(df_sales.columns))

print("COLUMNAS:", df_sales.columns)

ESQUEMA DEL DATASET
root
 |-- ProductID: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- Units: integer (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Country: string (nullable = true)

PRIMEROS 5 REGISTROS
+---------+---------+---------------+-----+-------+-------+
|ProductID|Date     |Zip            |Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|725      |1/15/1999|41540          |1    |115.5  |Germany|
|787      |6/6/2002 |41540          |1    |314.9  |Germany|
|788      |6/6/2002 |41540          |1    |314.9  |Germany|
|940      |1/15/1999|22587          |1    |687.7  |Germany|
|396      |1/15/1999|22587          |1    |857.1  |Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 5 rows
TOTAL DE REGISTROS: 841147
TOTAL DE COLUMNAS: 6
COLUMNAS: ['ProductID', 'Date', 'Zip', 'Units', 'Revenue', 'Country']


## 2. Ingresos totales por país

Se agrupan los registros por país y se calcula
la suma de los ingresos generados por las ventas.
Los resultados se ordenan de mayor a menor para
identificar los países con mayores ingresos.

In [12]:
# Calcular ingresos totales por país
ingresos_pais = df_sales.groupBy("Country").agg(
    F.sum("Revenue").cast("decimal(20,2)").alias("ingresos_totales")
).orderBy(F.desc("ingresos_totales"))

print("INGRESOS TOTALES POR PAIS")
ingresos_pais.show(100, truncate=False)

print("TOTAL DE PAISES:", ingresos_pais.count())

print("PAIS CON MAYORES INGRESOS")
ingresos_pais.show(1, truncate=False)

INGRESOS TOTALES POR PAIS
+-------+----------------+
|Country|ingresos_totales|
+-------+----------------+
|France |148490193.30    |
|Germany|131930510.10    |
|Mexico |103645802.60    |
|Canada |34114930.90     |
+-------+----------------+

TOTAL DE PAISES: 4
PAIS CON MAYORES INGRESOS
+-------+----------------+
|Country|ingresos_totales|
+-------+----------------+
|France |148490193.30    |
+-------+----------------+
only showing top 1 row


## 3. Ventas de Alemania con más de una unidad

Se filtran las ventas realizadas en Alemania
cuya cantidad de unidades vendidas es superior a uno.

In [13]:
ventas_alemania = df_sales.filter(
    (F.col("Country") == "Germany") &
    (F.col("Units") > 1)
)

print("VENTAS DE ALEMANIA CON MAS DE UNA UNIDAD")
ventas_alemania.show(10, truncate=False)

print("TOTAL DE REGISTROS FILTRADOS:",
      ventas_alemania.count())

VENTAS DE ALEMANIA CON MAS DE UNA UNIDAD
+---------+---------+---------------+-----+-------+-------+
|ProductID|Date     |Zip            |Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|901      |2/15/1999|13587          |2    |818.9  |Germany|
|676      |1/15/1999|21107          |2    |1603.9 |Germany|
|604      |4/10/2003|10243          |2    |989.6  |Germany|
|491      |3/15/1999|10319          |2    |1630.7 |Germany|
|910      |3/15/1999|10435          |2    |829.4  |Germany|
|645      |3/15/1999|10435          |2    |1493.1 |Germany|
|862      |4/10/2003|10435          |2    |608.9  |Germany|
|499      |2/15/1999|10435          |2    |1692.6 |Germany|
|631      |2/15/1999|40880          |2    |692.5  |Germany|
|635      |6/3/2002 |12627          |2    |1716.2 |Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 10 rows
TOTAL DE REGISTROS FILTRADOS: 7426


## 4. Código postal con mayor número de ventas en Alemania

Se filtran las ventas correspondientes a Alemania,
se limpian los códigos postales y se agrupan los
registros para identificar el código postal con
mayor cantidad de ventas.

In [14]:
# Filtrar todas las ventas de Alemania
alemania = df_sales.filter(
    F.col("Country") == "Germany"
)

# Limpiar espacios en los códigos postales
alemania = alemania.withColumn(
    "Zip", F.trim(F.col("Zip"))
)

# Agrupar por código postal y contar ventas
ventas_zip = alemania.groupBy("Zip").agg(
    F.count("*").alias("total_ventas")
).orderBy(F.desc("total_ventas"))

print("TOP 10 CODIGOS POSTALES DE ALEMANIA")
ventas_zip.show(10, truncate=False)

print("CODIGO POSTAL CON MAYOR NUMERO DE VENTAS")
ventas_zip.show(1, truncate=False)

TOP 10 CODIGOS POSTALES DE ALEMANIA
+-----+------------+
|Zip  |total_ventas|
+-----+------------+
|50169|2130        |
|52538|2007        |
|40597|1980        |
|20251|1828        |
|22111|1780        |
|12629|1772        |
|12351|1710        |
|22049|1683        |
|52428|1668        |
|32051|1627        |
+-----+------------+
only showing top 10 rows
CODIGO POSTAL CON MAYOR NUMERO DE VENTAS
+-----+------------+
|Zip  |total_ventas|
+-----+------------+
|50169|2130        |
+-----+------------+
only showing top 1 row


## 5. Clasificación de ventas

Se agrega una columna denominada Clasificacion
mediante una condición que permite identificar
las ventas con ingresos superiores a 500 como
Alta y las demás como Normal.

In [15]:
# Clasificar las ventas según sus ingresos
df_clasificado = df_sales.withColumn(
    "Clasificacion",
    F.when(F.col("Revenue") > 500, "Alta")
    .otherwise("Normal")
)

print("MUESTRA DE VENTAS CLASIFICADAS")
df_clasificado.show(10, truncate=False)

# Contar las ventas de cada clasificación
print("TOTAL DE VENTAS POR CLASIFICACION")

df_clasificado.groupBy("Clasificacion").count().show()

print("TOTAL DE REGISTROS:", df_clasificado.count())

MUESTRA DE VENTAS CLASIFICADAS
+---------+---------+---------------+-----+-------+-------+-------------+
|ProductID|Date     |Zip            |Units|Revenue|Country|Clasificacion|
+---------+---------+---------------+-----+-------+-------+-------------+
|725      |1/15/1999|41540          |1    |115.5  |Germany|Normal       |
|787      |6/6/2002 |41540          |1    |314.9  |Germany|Normal       |
|788      |6/6/2002 |41540          |1    |314.9  |Germany|Normal       |
|940      |1/15/1999|22587          |1    |687.7  |Germany|Alta         |
|396      |1/15/1999|22587          |1    |857.1  |Germany|Alta         |
|734      |4/10/2003|22587          |1    |330.7  |Germany|Normal       |
|769      |2/15/1999|22587          |1    |257.2  |Germany|Normal       |
|499      |1/15/1999|12555          |1    |846.3  |Germany|Alta         |
|2254     |1/15/1999|40217          |1    |57.7   |Germany|Normal       |
|31       |5/31/2002|40217          |1    |761.2  |Germany|Alta         |
+------

## 6. Bonus: Ingresos promedio por año

Se convierte la columna Date de texto a fecha
utilizando el formato M/d/yyyy. Posteriormente,
se extrae el año de cada venta y se calcula
el promedio de ingresos por año.

In [16]:
# Convertir texto a fecha
df_fechas = df_sales.withColumn(
    "Fecha",
    F.to_date(F.col("Date"), "M/d/yyyy")
)

# Extraer el año
df_fechas = df_fechas.withColumn(
    "Anio", F.year(F.col("Fecha"))
)

print("MUESTRA DE FECHAS CONVERTIDAS")
df_fechas.select("Date", "Fecha", "Anio").show(10)

# Verificar fechas no válidas
print("FECHAS NULAS:",
      df_fechas.filter(F.col("Fecha").isNull()).count())

# Calcular ingresos promedio por año
ingresos_anio = df_fechas.groupBy("Anio").agg(
    F.avg("Revenue").cast("decimal(20,2)")
    .alias("ingreso_promedio")
).orderBy("Anio")

print("INGRESOS PROMEDIO POR ANIO")
ingresos_anio.show(100, truncate=False)

MUESTRA DE FECHAS CONVERTIDAS
+---------+----------+----+
|     Date|     Fecha|Anio|
+---------+----------+----+
|1/15/1999|1999-01-15|1999|
| 6/6/2002|2002-06-06|2002|
| 6/6/2002|2002-06-06|2002|
|1/15/1999|1999-01-15|1999|
|1/15/1999|1999-01-15|1999|
|4/10/2003|2003-04-10|2003|
|2/15/1999|1999-02-15|1999|
|1/15/1999|1999-01-15|1999|
|1/15/1999|1999-01-15|1999|
|5/31/2002|2002-05-31|2002|
+---------+----------+----+
only showing top 10 rows
FECHAS NULAS: 0
INGRESOS PROMEDIO POR ANIO
+----+----------------+
|Anio|ingreso_promedio|
+----+----------------+
|1999|420.59          |
|2000|401.73          |
|2001|399.33          |
|2002|417.90          |
|2003|423.92          |
|2004|437.43          |
|2005|457.53          |
|2006|500.54          |
|2007|514.12          |
|2008|520.91          |
|2009|579.83          |
|2010|625.87          |
|2011|652.38          |
|2012|686.16          |
|2013|712.75          |
|2014|653.87          |
|2015|628.77          |
+----+----------------+



# Conclusiones

Durante esta práctica se utilizaron los DataFrames
de Apache Spark para realizar operaciones de
creación, filtrado, transformación, agregación,
funciones de ventana y consultas SQL.

En el proyecto final se analizaron 841,147 registros
de ventas correspondientes a cuatro países.

Francia presentó los mayores ingresos, con un total
de 148,490,193.30. En Alemania se identificaron
7,426 registros de ventas con más de una unidad,
y el código postal 50169 presentó la mayor cantidad
de ventas, con 2,130 registros.

La clasificación permitió identificar 507,530
ventas normales y 333,617 ventas altas.

Finalmente, se analizaron los ingresos promedio
por año, identificando que 2013 presentó el
mayor promedio, con un valor de 712.75.

Esta práctica permitió comprender la utilidad
de Apache Spark para procesar grandes volúmenes
de datos y obtener información relevante mediante
operaciones de DataFrames.

# Referencias

- Apache Software Foundation. PySpark SQL API.
  https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

- Apache Software Foundation. Spark SQL,
  DataFrames and Datasets Guide.
  https://spark.apache.org/docs/latest/sql-programming-guide.html

- Dataset utilizado: pdi_sales.csv.
  https://github.com/josepgarcia/datos/blob/main/pdi_sales.csv

In [17]:
spark.stop()
print("Practica finalizada correctamente.")

Practica finalizada correctamente.
